# Phase 1 — Load and understand the dataset

# Restaurant Data Cleaning

This notebook inspects, cleans, validates, and prepares the expanded restaurant dataset for Power BI.

## Import Libraries

In [66]:
from pathlib import Path
import pandas as pd

## Load Expanded Dataset

In [67]:
PROJECT_ROOT = Path.cwd().parent

excel_path = PROJECT_ROOT / "data" / "Restaurant_Expanded.xlsx"

excel_path

WindowsPath('e:/Projects/restaurant-powerbi-dashboard/restaurant-powerbi-dashboard/data/Restaurant_Expanded.xlsx')

In [68]:
excel_file = pd.ExcelFile(excel_path)

excel_file.sheet_names

['Dishes',
 'Events',
 'EventsLocations',
 'Orders',
 'OrdersDishes',
 'CustomersEvents',
 'CustomersDishes',
 'Customers',
 'Reservations']

## Load All Tables

In [69]:
customers = pd.read_excel(excel_path, sheet_name="Customers")
dishes = pd.read_excel(excel_path, sheet_name="Dishes")
orders = pd.read_excel(excel_path, sheet_name="Orders")
orders_dishes = pd.read_excel(excel_path, sheet_name="OrdersDishes")
reservations = pd.read_excel(excel_path, sheet_name="Reservations")
events = pd.read_excel(excel_path, sheet_name="Events")
events_locations = pd.read_excel(excel_path, sheet_name="EventsLocations")
customers_events = pd.read_excel(excel_path, sheet_name="CustomersEvents")
customers_dishes = pd.read_excel(excel_path, sheet_name="CustomersDishes")

## Dataset Overview

In [70]:
tables = {
    "Customers": customers,
    "Dishes": dishes,
    "Orders": orders,
    "OrdersDishes": orders_dishes,
    "Reservations": reservations,
    "Events": events,
    "EventsLocations": events_locations,
    "CustomersEvents": customers_events,
    "CustomersDishes": customers_dishes
}

In [71]:
overview = []

for name, df in tables.items():
    overview.append({
        "Table": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1]
    })

overview_df = pd.DataFrame(overview)

overview_df

,Table,Rows,Columns
0,Customers,100,10
1,Dishes,22,4
2,Orders,1500,3
3,OrdersDishes,4401,3
4,Reservations,505,4
5,Events,15,5
6,EventsLocations,5,3
7,CustomersEvents,306,3
8,CustomersDishes,255,3


# Phase 2 — Inspect data quality

## 1. Check Missing Values

In [72]:
missing_summary = []

for name, df in tables.items():
    missing_summary.append({
        "Table": name,
        "MissingValues": df.isnull().sum().sum()
    })

missing_df = pd.DataFrame(missing_summary)

missing_df

,Table,MissingValues
0,Customers,0
1,Dishes,0
2,Orders,0
3,OrdersDishes,0
4,Reservations,8
5,Events,0
6,EventsLocations,0
7,CustomersEvents,0
8,CustomersDishes,0


## 2. Check Duplicate Rows

In [73]:
duplicate_summary = []

for name, df in tables.items():
    duplicate_summary.append({
        "Table": name,
        "DuplicateRows": df.duplicated().sum()
    })

duplicate_df = pd.DataFrame(duplicate_summary)

duplicate_df

,Table,DuplicateRows
0,Customers,0
1,Dishes,0
2,Orders,0
3,OrdersDishes,0
4,Reservations,5
5,Events,0
6,EventsLocations,0
7,CustomersEvents,6
8,CustomersDishes,5


## 3. Primary Key Uniqueness

In [74]:
primary_keys = {
    "Customers": "CustomerID",
    "Dishes": "DishID",
    "Orders": "OrderID",
    "OrdersDishes": "OrdersDishesID",
    "Reservations": "ReservationID",
    "Events": "EventID",
    "EventsLocations": "id",
    "CustomersEvents": "CustomersEventsID",
    "CustomersDishes": "CustomersDishesID"
}

In [75]:
key_summary = []

for table_name, key_column in primary_keys.items():
    df = tables[table_name]

    key_summary.append({
        "Table": table_name,
        "PrimaryKey": key_column,
        "IsUnique": df[key_column].is_unique,
        "DuplicateKeys": df[key_column].duplicated().sum()
    })

key_df = pd.DataFrame(key_summary)

key_df

,Table,PrimaryKey,IsUnique,DuplicateKeys
0,Customers,CustomerID,True,0
1,Dishes,DishID,True,0
2,Orders,OrderID,True,0
3,OrdersDishes,OrdersDishesID,True,0
4,Reservations,ReservationID,False,5
5,Events,EventID,True,0
6,EventsLocations,id,True,0
7,CustomersEvents,CustomersEventsID,False,6
8,CustomersDishes,CustomersDishesID,False,5


## 4. Data Types

In [76]:
dtype_summary = []

for name, df in tables.items():
    for column in df.columns:
        dtype_summary.append({
            "Table": name,
            "Column": column,
            "DataType": str(df[column].dtype)
        })

dtype_df = pd.DataFrame(dtype_summary)

dtype_df

,Table,Column,DataType
0,Customers,CustomerID,int64
1,Customers,FirstName,str
2,Customers,LastName,str
3,Customers,Email,str
4,Customers,Address,str
5,Customers,City,str
6,Customers,State,str
7,Customers,Phone,str
8,Customers,Birthday,datetime64[us]
9,Customers,FavoriteDish,int64


## 5. Value Validation

In [77]:
print("Dish prices:")
print(dishes["Price"].describe())

print("\nReservation party sizes:")
print(reservations["PartySize"].describe())

print("\nOrder date range:")
print(orders["OrderDate"].min(), "to", orders["OrderDate"].max())

print("\nReservation date range:")
print(reservations["Date"].min(), "to", reservations["Date"].max())

print("\nEvent date range:")
print(events["Date"].min(), "to", events["Date"].max())

Dish prices:
count    22.000000
mean      8.042273
std       1.860381
min       4.000000
25%       7.000000
50%       8.500000
75%       9.990000
max       9.990000
Name: Price, dtype: float64

Reservation party sizes:
count    497.000000
mean       3.235412
std        1.471131
min        1.000000
25%        2.000000
50%        3.000000
75%        4.000000
max        6.000000
Name: PartySize, dtype: float64

Order date range:
2019-02-01 17:32:00 to 2019-12-31 19:30:00

Reservation date range:
2019-02-05 14:00:00 to 2019-12-31 18:00:00

Event date range:
2014-01-01 18:00:00 to 2019-10-31 17:00:00


In [78]:
print("Invalid dish prices:", (dishes["Price"] <= 0).sum())

print(
    "Invalid party sizes:",
    (reservations["PartySize"].dropna() <= 0).sum()
)

Invalid dish prices: 0
Invalid party sizes: 0


## 6. Foreign Key Validation

In [79]:
foreign_key_checks = [
    ("Orders", "CustomerID", "Customers", "CustomerID"),
    ("OrdersDishes", "OrderID", "Orders", "OrderID"),
    ("OrdersDishes", "DishID", "Dishes", "DishID"),
    ("Reservations", "CustomerID", "Customers", "CustomerID"),
    ("CustomersEvents", "CustomerID", "Customers", "CustomerID"),
    ("CustomersEvents", "EventID", "Events", "EventID"),
    ("CustomersDishes", "CustomerID", "Customers", "CustomerID"),
    ("CustomersDishes", "DishID", "Dishes", "DishID"),
    ("Customers", "FavoriteDish", "Dishes", "DishID")
]

In [80]:
foreign_key_summary = []

for child_table, child_column, parent_table, parent_column in foreign_key_checks:

    child_df = tables[child_table]
    parent_df = tables[parent_table]

    valid_mask = child_df[child_column].isin(parent_df[parent_column])

    foreign_key_summary.append({
        "Relationship": f"{child_table}.{child_column} → {parent_table}.{parent_column}",
        "Valid": valid_mask.all(),
        "InvalidRows": (~valid_mask).sum()
    })

foreign_key_df = pd.DataFrame(foreign_key_summary)

foreign_key_df

,Relationship,Valid,InvalidRows
0,Orders.CustomerID → Customers.CustomerID,True,0
1,OrdersDishes.OrderID → Orders.OrderID,True,0
2,OrdersDishes.DishID → Dishes.DishID,True,0
3,Reservations.CustomerID → Customers.CustomerID,True,0
4,CustomersEvents.CustomerID → Customers.CustomerID,True,0
5,CustomersEvents.EventID → Events.EventID,True,0
6,CustomersDishes.CustomerID → Customers.CustomerID,True,0
7,CustomersDishes.DishID → Dishes.DishID,True,0
8,Customers.FavoriteDish → Dishes.DishID,True,0


# Phase 3 — Data Cleaning

After identifying the data quality problems, the next step is to clean them in a logical order:
1. Handle missing values
2. Remove duplicate rows
3. Correct data types
4. Revalidate keys and relationships

## 1. Handle Missing Values

The missing values are all in `Reservations.PartySize`.

Since party size is a numerical variable and the missing reservations are otherwise valid, the missing values will be filled using the median party size.

In [81]:
party_size_median = reservations["PartySize"].median()

party_size_median

np.float64(3.0)

In [82]:
reservations["PartySize"] = reservations["PartySize"].fillna(party_size_median)

In [83]:
reservations["PartySize"].isnull().sum()

np.int64(0)

## 2. Remove Duplicate Rows

Duplicate rows were found in `Reservations`, `CustomersEvents`, and `CustomersDishes`.

The duplicate records will be removed, keeping the first occurrence.

In [84]:
reservations = reservations.drop_duplicates()

customers_events = customers_events.drop_duplicates()

customers_dishes = customers_dishes.drop_duplicates()

In [85]:
print("Reservations duplicates:", reservations.duplicated().sum())
print("CustomersEvents duplicates:", customers_events.duplicated().sum())
print("CustomersDishes duplicates:", customers_dishes.duplicated().sum())

Reservations duplicates: 0
CustomersEvents duplicates: 0
CustomersDishes duplicates: 0


## 3. Fix Data Types

After filling the missing `PartySize` values, the column can now be converted from `float64` to `int64` because party size represents a whole number of people.

In [86]:
reservations["PartySize"] = reservations["PartySize"].astype(int)

# Phase 4 — Validation

After cleaning, the dataset is checked again to confirm that:
- no missing values remain
- no duplicate rows remain
- primary keys are unique
- foreign-key relationships are still valid

#### Update Cleaned Tables

In [87]:
tables["Reservations"] = reservations
tables["CustomersEvents"] = customers_events
tables["CustomersDishes"] = customers_dishes

#### Final quality summary

In [88]:
final_quality = []

for name, df in tables.items():
    key_column = primary_keys[name]

    final_quality.append({
        "Table": name,
        "Rows": df.shape[0],
        "MissingValues": df.isnull().sum().sum(),
        "DuplicateRows": df.duplicated().sum(),
        "PrimaryKeyUnique": df[key_column].is_unique
    })

final_quality_df = pd.DataFrame(final_quality)

final_quality_df

,Table,Rows,MissingValues,DuplicateRows,PrimaryKeyUnique
0,Customers,100,0,0,True
1,Dishes,22,0,0,True
2,Orders,1500,0,0,True
3,OrdersDishes,4401,0,0,True
4,Reservations,500,0,0,True
5,Events,15,0,0,True
6,EventsLocations,5,0,0,True
7,CustomersEvents,300,0,0,True
8,CustomersDishes,250,0,0,True


#### Relationship Validation

In [89]:
foreign_key_summary = []

for child_table, child_column, parent_table, parent_column in foreign_key_checks:

    child_df = tables[child_table]
    parent_df = tables[parent_table]

    valid_mask = child_df[child_column].isin(parent_df[parent_column])

    foreign_key_summary.append({
        "Relationship": f"{child_table}.{child_column} → {parent_table}.{parent_column}",
        "Valid": valid_mask.all(),
        "InvalidRows": (~valid_mask).sum()
    })

final_fk_df = pd.DataFrame(foreign_key_summary)

final_fk_df

,Relationship,Valid,InvalidRows
0,Orders.CustomerID → Customers.CustomerID,True,0
1,OrdersDishes.OrderID → Orders.OrderID,True,0
2,OrdersDishes.DishID → Dishes.DishID,True,0
3,Reservations.CustomerID → Customers.CustomerID,True,0
4,CustomersEvents.CustomerID → Customers.CustomerID,True,0
5,CustomersEvents.EventID → Events.EventID,True,0
6,CustomersDishes.CustomerID → Customers.CustomerID,True,0
7,CustomersDishes.DishID → Dishes.DishID,True,0
8,Customers.FavoriteDish → Dishes.DishID,True,0


# Phase 5 — Save the clean dataset

The cleaned tables are exported to a new Excel workbook for use in Power BI. The original and expanded datasets are kept unchanged.

In [90]:
output_path = PROJECT_ROOT / "data" / "processed" / "Restaurant_Cleaned.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sheet_name, df in tables.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

output_path

WindowsPath('e:/Projects/restaurant-powerbi-dashboard/restaurant-powerbi-dashboard/data/processed/Restaurant_Cleaned.xlsx')